# Part C (PM) — Interview Ready
**Day 33 | PM Session | Week 6**

Three interview-style questions: high-dimensional data strategy, model selection utility function, and SVM overfitting diagnosis.

---
## Q1 — Conceptual: 100 Features, 50 Samples

**Question:** You have a dataset with 100 features and 50 samples. Which algorithms from this week would work best and why? Which would fail?

### Answer

This is a classic **p >> n** (high-dimensional, low-sample) scenario. Here's the analysis:

| Algorithm | Would it work? | Reasoning |
|-----------|:--------------:|----------|
| **Logistic Regression (L1/L2)** | ✅ Best choice | Regularisation (L1 especially) handles p>>n by shrinking/zeroing irrelevant features. With L1 (Lasso), acts as feature selector. |
| **SVM (Linear kernel)** | ✅ Good | Maximises margin — effective in high-dim. Linear SVC with small C handles overfitting. |
| **SVM (RBF kernel)** | ⚠️ Risky | RBF in 100 dims with 50 samples can overfit unless gamma is very small. Use linear kernel instead. |
| **Naive Bayes** | ✅ Good baseline | Independence assumption is actually helpful here — no complex interactions to overfit. Fast. |
| **Decision Tree** | ❌ Fails | With 100 features and 50 samples, DT will find a perfect split and overfit to 100% train accuracy, near-random on test. |
| **Random Forest** | ⚠️ Mediocre | Bagging helps, but each tree still faces 100 features on 50 samples. Feature importance becomes noisy. Need heavy min_samples constraints. |
| **KNN** | ❌ Fails | Curse of dimensionality: in 100D, all points are approximately equidistant. KNN's distance metric becomes meaningless. |
| **Gradient Boosting / XGBoost** | ❌ Fails | Boosting needs many samples to find reliable splits. Will memorise the 50 training samples. Even with regularisation, performance is poor. |

### Recommended Strategy
1. **First**: Apply dimensionality reduction (PCA, keeping 90% variance) or feature selection (SelectKBest, ANOVA F-test).
2. **Model**: Logistic Regression with L1 penalty + strong cross-validation (LOOCV or 5-fold).
3. **Alternative**: Linear SVM (`LinearSVC`) — equivalent but sometimes faster.
4. **Evaluate with**: Stratified K-fold (k=5 or LOOCV) since each fold has very few samples.

> **Rule of thumb**: You need at least 5-10 samples per feature for reliable parameter estimation. With 50 samples and 100 features, always use regularisation and prefer sparse linear models.

---
## Q2 — Coding: `model_selection_report`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

# sklearn models for demo
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB


def model_selection_report(
    X: np.ndarray,
    y: np.ndarray,
    models_dict: dict,
    cv: int = 5,
    scoring: str = 'accuracy',
    scale: bool = True,
    alpha: float = 0.05
) -> pd.DataFrame:
    """
    Run cross-validation for multiple models and return a formatted comparison DataFrame.

    Parameters
    ----------
    X           : Feature matrix
    y           : Target vector
    models_dict : {model_name: sklearn_estimator}
    cv          : Number of CV folds
    scoring     : sklearn scoring string
    scale       : Whether to apply StandardScaler inside each fold
    alpha       : Significance level for paired t-tests

    Returns
    -------
    DataFrame ranked by mean CV score, with statistical best model identified.
    """
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    all_scores  = {}   # {name: np.array of fold scores}

    for name, model in models_dict.items():
        if scale:
            # Wrap in pipeline to prevent data leakage across folds
            estimator = Pipeline([('scaler', StandardScaler()), ('model', model)])
        else:
            estimator = model
        scores = cross_val_score(estimator, X, y, cv=cv_splitter,
                                 scoring=scoring, n_jobs=-1)
        all_scores[name] = scores

    # Build summary DataFrame
    rows = []
    for name, scores in all_scores.items():
        rows.append({
            'Model'    : name,
            'Mean'     : scores.mean(),
            'Std'      : scores.std(),
            'Min'      : scores.min(),
            'Max'      : scores.max(),
            '95% CI Lo': scores.mean() - 1.96 * scores.std() / np.sqrt(cv),
            '95% CI Hi': scores.mean() + 1.96 * scores.std() / np.sqrt(cv),
        })

    df = pd.DataFrame(rows).sort_values('Mean', ascending=False).reset_index(drop=True)
    df.index += 1  # rank from 1

    # Paired t-test: best model vs every other
    best_name   = df.iloc[0]['Model']
    best_scores = all_scores[best_name]
    sig_flags   = []

    for name in df['Model']:
        if name == best_name:
            sig_flags.append('— (best)')
        else:
            _, p = stats.ttest_rel(best_scores, all_scores[name])
            flag = f'p={p:.3f}' + (' ✅ sig.' if p < alpha else ' ❌ not sig.')
            sig_flags.append(flag)

    df['vs Best (paired t)'] = sig_flags

    print(f'Statistical best model: {best_name}  (mean {df.iloc[0]["Mean"]:.4f})')
    print(f'Significance threshold: α={alpha}')
    print()
    return df


# --- Demo ---
data = load_breast_cancer()
X, y = data.data, data.target

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)'          : SVC(kernel='rbf', C=10, gamma='scale'),
    'KNN'                : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'        : GaussianNB(),
}

report = model_selection_report(X, y, models, cv=5)
pd.set_option('display.float_format', '{:.4f}'.format)
display(report)

---
## Q3 — Analyze: SVM(RBF) Train=1.0, Test=0.52

```python
# Scenario: train accuracy 1.0, test accuracy 0.52
svm = SVC(kernel='rbf', C=100, gamma=10)
svm.fit(X_train_scaled, y_train)
# train: 1.0  |  test: 0.52
```

### Root Cause: Severe Overfitting

Train=1.0 / Test=0.52 is a textbook **overfitting** signature. The model has memorised the training set and fails to generalise.

### 3 Specific Fixes

**Fix 1 — Reduce C (soften the margin)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Simulate scenario: small dataset, many features
X, y = make_classification(n_samples=200, n_features=20, n_informative=5,
                            random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# Broken model
svm_broken = SVC(kernel='rbf', C=1000, gamma=10)
svm_broken.fit(X_tr_sc, y_tr)
print(f'Broken — Train: {svm_broken.score(X_tr_sc, y_tr):.4f} | Test: {svm_broken.score(X_te_sc, y_te):.4f}')

# Fix 1: Lower C
svm_fix1 = SVC(kernel='rbf', C=1.0, gamma=10)
svm_fix1.fit(X_tr_sc, y_tr)
print(f'Fix 1 (C=1)  — Train: {svm_fix1.score(X_tr_sc, y_tr):.4f} | Test: {svm_fix1.score(X_te_sc, y_te):.4f}')

# Fix 2: Lower gamma (wider kernel → smoother boundary)
svm_fix2 = SVC(kernel='rbf', C=10, gamma=0.01)
svm_fix2.fit(X_tr_sc, y_tr)
print(f'Fix 2 (γ=0.01) — Train: {svm_fix2.score(X_tr_sc, y_tr):.4f} | Test: {svm_fix2.score(X_te_sc, y_te):.4f}')

# Fix 3: GridSearchCV to find best C and gamma jointly
from sklearn.model_selection import GridSearchCV
pipe = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf'))])
grid = GridSearchCV(pipe,
                    param_grid={'svm__C': [0.1, 1, 10], 'svm__gamma': [0.001, 0.01, 0.1]},
                    cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr, y_tr)
print(f'Fix 3 (GridCV) — Test: {grid.score(X_te, y_te):.4f} | Best params: {grid.best_params_}')

### Summary of Fixes

| Fix | Action | Why it works |
|-----|--------|--------------|
| **1. Reduce C** | `C=100 → C=1` | Allows more margin violations (soft margin); less memorisation |
| **2. Reduce gamma** | `gamma=10 → gamma='scale'` or `0.01` | Wider Gaussian kernel = smoother decision boundary; less influenced by individual points |
| **3. GridSearchCV** | Tune C and gamma jointly with 5-fold CV | Finds the bias-variance sweet spot empirically; eliminates guesswork |

> **Root cause in one sentence**: `C=100, gamma=10` creates an extremely wiggly, overfit boundary — each training point gets its own tiny local region. Reduce either (or both) to smooth out the boundary and generalise.